# KNN

### X and y

In [1]:
import pandas as pd

# Load CSV
features_df = pd.read_csv("hyperaktiv_with_controls/features.csv", sep=";")

# Set participant ID as index
features_df = features_df.set_index("ID")

# This is your X matrix
X = features_df.copy()


In [2]:
print(X.shape)
print(X.isna().sum().sum(), "NaNs total")
print((X.var() == 0).sum(), "zero-variance features")

(116, 787)
696 NaNs total
47 zero-variance features


In [3]:
# Drop columns with any NaNs
X = X.dropna(axis=1)

# Drop zero-variance features
X = X.loc[:, X.var() > 0]

# Optional: reduce precision to save memory
# X = X.astype("float32")

In [4]:
import pandas as pd

# Load patient info
info = pd.read_csv("hyperaktiv_with_controls/patient_info.csv", sep=";")

# Keep only relevant columns
info = info[["ID", "ADHD", "ADD"]]

# Align with X (important!)
info = info.set_index("ID").loc[X.index]

# Create multiclass target
def build_multiclass_label(row):
    if row["ADHD"] == 0:
        return 0  # No ADHD
    elif row["ADHD"] == 1 and row["ADD"] == 0:
        return 1  # ADHD without inattentive subtype
    elif row["ADHD"] == 1 and row["ADD"] == 1:
        return 2  # ADHD with inattentive subtype

y = info.apply(build_multiclass_label, axis=1)

# Convert to integer type
y = y.astype(int)


In [5]:
print(y.value_counts())
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Index aligned:", X.index.equals(y.index))

0    71
2    23
1    22
Name: count, dtype: int64
X shape: (116, 734)
y shape: (116,)
Index aligned: True


In [6]:
from tsfresh.feature_selection import select_features

X_selected = select_features(
    X,
    y,
    fdr_level=0.05
)

print("Original features:", X.shape[1])
print("Selected features:", X_selected.shape[1])

Original features: 734
Selected features: 52


### Split Data

In [7]:
from sklearn.model_selection import train_test_split

# 80/20 split
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_selected,
    y,
    test_size=0.2,
    random_state=5,
    stratify=y
)

print(f"Train/Val samples: {len(X_train_val)}")
print(f"Test samples: {len(X_test)}")


Train/Val samples: 92
Test samples: 24


### KNN

In [8]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, f1_score

# KNN parameter grid
n_neighbors_values = [3, 5, 7, 9, 11]
weight_values = ["uniform", "distance"]
metric_values = ["euclidean", "manhattan"]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=5)

results = []
all_reports = []

for n_neighbors in n_neighbors_values:
    for weights in weight_values:
        for metric in metric_values:

            fold_f1 = []
            fold_reports = []

            for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_val, y_train_val), 1):
                X_train, X_val = X_train_val.iloc[train_idx], X_train_val.iloc[val_idx]
                y_train, y_val = y_train_val.iloc[train_idx], y_train_val.iloc[val_idx]

                pipeline = Pipeline([
                    ("scaler", StandardScaler()),
                    ("knn", KNeighborsClassifier(
                        n_neighbors=n_neighbors,
                        weights=weights,
                        metric=metric
                    ))
                ])

                pipeline.fit(X_train, y_train)
                y_pred = pipeline.predict(X_val)

                f1 = f1_score(y_val, y_pred, average="macro")
                fold_f1.append(f1)

                report = classification_report(
                    y_val,
                    y_pred,
                    output_dict=True,
                    zero_division=0
                )
                fold_reports.append(report)

            # Store summary results
            results.append({
                "n_neighbors": n_neighbors,
                "weights": weights,
                "metric": metric,
                "f1_macro_mean": np.mean(fold_f1),
                "f1_macro_std": np.std(fold_f1)
            })

            # Store full reports (optional but useful)
            all_reports.append({
                "params": (n_neighbors, weights, metric),
                "reports": fold_reports
            })

results_df = pd.DataFrame(results).sort_values(
    by="f1_macro_mean",
    ascending=False
)

results_df


,n_neighbors,weights,metric,f1_macro_mean,f1_macro_std
1,3,uniform,manhattan,0.557123,0.114954
7,5,distance,manhattan,0.540410,0.099039
3,3,distance,manhattan,0.539360,0.114572
11,7,distance,manhattan,0.536383,0.060667
5,5,uniform,manhattan,0.526204,0.086536
2,3,distance,euclidean,0.521993,0.082338
9,7,uniform,manhattan,0.521408,0.083679
6,5,distance,euclidean,0.491862,0.082429
12,9,uniform,euclidean,0.488677,0.115472
17,11,uniform,manhattan,0.484013,0.106293


### Best Parameters

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, f1_score, balanced_accuracy_score, confusion_matrix

# Best params from CV
best_n_neighbors = 7
best_weights = "distance"
best_metric = "manhattan"

final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(
        n_neighbors=best_n_neighbors,
        weights=best_weights,
        metric=best_metric
    ))
])

# Train on full train+val
final_model.fit(X_train_val, y_train_val)

# Evaluate on test set
y_test_pred = final_model.predict(X_test)

print("Macro F1:", f1_score(y_test, y_test_pred, average="macro"))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_test_pred))

print("=== Final Test Results (KNN) ===")
print(classification_report(y_test, y_test_pred))

print("\nTest Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))


Macro F1: 0.35873015873015873
Balanced Accuracy: 0.37222222222222223
=== Final Test Results (KNN) ===
              precision    recall  f1-score   support

           0       0.65      0.87      0.74        15
           1       0.50      0.25      0.33         4
           2       0.00      0.00      0.00         5

    accuracy                           0.58        24
   macro avg       0.38      0.37      0.36        24
weighted avg       0.49      0.58      0.52        24


Test Confusion Matrix:
[[13  1  1]
 [ 2  1  1]
 [ 5  0  0]]
